In [1]:
!pip -q install sentence-transformers

In [2]:
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from tqdm.auto import tqdm

In [3]:
MASTER = "/kaggle/input/notebooks/shri7ul/03-feature-engineering-ipynb/master_feature_engineered.parquet"

master = pd.read_parquet(MASTER)

print(master.shape)

(35072, 122)


In [4]:
MODEL_PATH = "/kaggle/input/huggingface-models/sentence-transformers/all-MiniLM-L6-v2"

In [5]:
try:

    model = SentenceTransformer(MODEL_PATH)

except:

    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
student_embeddings = model.encode(

    master["student_text"].fillna("").tolist(),

    batch_size=64,

    show_progress_bar=True,

    normalize_embeddings=True,
)

Batches:   0%|          | 0/548 [00:00<?, ?it/s]

In [7]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd

MODEL_PATH = "/kaggle/input/huggingface-models/sentence-transformers/all-MiniLM-L6-v2"

try:
    model = SentenceTransformer(MODEL_PATH)
except:
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


unique_objectives = (
    master["learning_objective"]
    .fillna("")
    .unique()
)

print("Unique Objectives :", len(unique_objectives))

objective_embeddings = model.encode(
    unique_objectives,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Unique Objectives : 398


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

In [8]:
from sklearn.decomposition import PCA

N_COMPONENTS = 32

pca = PCA(
    n_components=N_COMPONENTS,
    random_state=42,
)

objective_embeddings_pca = pca.fit_transform(objective_embeddings)

print(objective_embeddings.shape)
print(objective_embeddings_pca.shape)

(398, 384)
(398, 32)


In [9]:
embedding_map = {
    obj: emb
    for obj, emb in zip(
        unique_objectives,
        objective_embeddings_pca
    )
}

In [10]:
objective_matrix = np.vstack(
    master["learning_objective"]
    .fillna("")
    .map(embedding_map)
)

objective_df = pd.DataFrame(
    objective_matrix,
    columns=[
        f"obj_emb_{i}"
        for i in range(objective_matrix.shape[1])
    ]
)

master = pd.concat(
    [
        master.reset_index(drop=True),
        objective_df.reset_index(drop=True)
    ],
    axis=1,
)

print(master.shape)

objective_df.head()

(35072, 154)


,obj_emb_0,obj_emb_1,obj_emb_2,obj_emb_3,obj_emb_4,obj_emb_5,obj_emb_6,obj_emb_7,obj_emb_8,obj_emb_9,...,obj_emb_22,obj_emb_23,obj_emb_24,obj_emb_25,obj_emb_26,obj_emb_27,obj_emb_28,obj_emb_29,obj_emb_30,obj_emb_31
0,0.297763,0.342341,-0.055806,-0.166520,0.193050,0.083793,-0.047799,-0.010479,0.018329,-0.043580,...,-0.036744,0.050522,-0.142229,-0.034154,-0.041571,0.096068,-0.106509,-0.002536,0.018403,-0.097688
1,0.298976,0.186476,0.156693,0.055692,0.267455,0.169488,0.091929,0.165543,0.074871,-0.232244,...,-0.084418,0.075249,-0.007156,-0.005594,-0.062537,0.049526,0.039313,-0.047111,0.009420,-0.103667
2,0.110943,-0.355513,-0.167190,-0.346841,-0.247081,-0.236838,0.164924,-0.065447,0.101506,-0.055301,...,0.062302,0.119097,0.062890,-0.044874,0.084869,-0.006300,-0.056318,0.054293,0.066928,0.005326
3,0.084921,-0.306909,-0.328299,-0.238743,-0.216909,-0.261822,0.014085,-0.000819,0.079435,-0.250121,...,0.073597,0.026813,-0.085554,0.098834,0.027008,-0.040452,-0.025348,0.007394,-0.001952,-0.020721
4,0.070673,0.175749,0.238238,-0.018993,-0.146282,-0.139526,-0.132248,0.212716,-0.023558,-0.031073,...,-0.030643,-0.070862,-0.079504,-0.059128,0.189735,-0.102673,0.121264,0.161540,-0.122004,-0.095329


In [11]:
display(
    master[
        [c for c in master.columns if c.startswith("obj_emb_")]
    ].head()
)

print("Embedding Features :", len([c for c in master.columns if c.startswith("obj_emb_")]))

,obj_emb_0,obj_emb_1,obj_emb_2,obj_emb_3,obj_emb_4,obj_emb_5,obj_emb_6,obj_emb_7,obj_emb_8,obj_emb_9,...,obj_emb_22,obj_emb_23,obj_emb_24,obj_emb_25,obj_emb_26,obj_emb_27,obj_emb_28,obj_emb_29,obj_emb_30,obj_emb_31
0,0.297763,0.342341,-0.055806,-0.166520,0.193050,0.083793,-0.047799,-0.010479,0.018329,-0.043580,...,-0.036744,0.050522,-0.142229,-0.034154,-0.041571,0.096068,-0.106509,-0.002536,0.018403,-0.097688
1,0.298976,0.186476,0.156693,0.055692,0.267455,0.169488,0.091929,0.165543,0.074871,-0.232244,...,-0.084418,0.075249,-0.007156,-0.005594,-0.062537,0.049526,0.039313,-0.047111,0.009420,-0.103667
2,0.110943,-0.355513,-0.167190,-0.346841,-0.247081,-0.236838,0.164924,-0.065447,0.101506,-0.055301,...,0.062302,0.119097,0.062890,-0.044874,0.084869,-0.006300,-0.056318,0.054293,0.066928,0.005326
3,0.084921,-0.306909,-0.328299,-0.238743,-0.216909,-0.261822,0.014085,-0.000819,0.079435,-0.250121,...,0.073597,0.026813,-0.085554,0.098834,0.027008,-0.040452,-0.025348,0.007394,-0.001952,-0.020721
4,0.070673,0.175749,0.238238,-0.018993,-0.146282,-0.139526,-0.132248,0.212716,-0.023558,-0.031073,...,-0.030643,-0.070862,-0.079504,-0.059128,0.189735,-0.102673,0.121264,0.161540,-0.122004,-0.095329


Embedding Features : 32


In [12]:
# ==========================================================
# Save Semantic Dataset
# ==========================================================

OUTPUT = "/kaggle/working/master_semantic.parquet"

master.to_parquet(
    OUTPUT,
    index=False,
)

print(master.shape)
print("Saved :", OUTPUT)

(35072, 154)
Saved : /kaggle/working/master_semantic.parquet


In [13]:
# ==========================================================
# Save PCA
# ==========================================================

import pickle

with open("/kaggle/working/objective_pca.pkl", "wb") as f:
    pickle.dump(pca, f)

print("✓ objective_pca.pkl saved")

✓ objective_pca.pkl saved


In [14]:
import os

print(os.listdir("/kaggle/working"))

['__notebook__.ipynb', 'master_semantic.parquet', 'objective_pca.pkl']


In [15]:
import json

semantic_config = {
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "embedding_dimension": 384,
    "pca_dimension": 32,
}

with open("/kaggle/working/semantic_config.json", "w") as f:
    json.dump(semantic_config, f, indent=4)

print("✓ semantic_config.json saved")

✓ semantic_config.json saved
